# Detectors for custom (interleaved) syndrome-extraction schedules

`tqecd` splits a circuit into *fragments* that each have the shape
`[leading resets] [computation] [trailing measurements]`. A depth-optimised
circuit can break that shape by *interleaving* a reset or a measurement with the
two-qubit gates. `tqecd.construction.annotate_detectors_automatically` handles
such circuits automatically: it reschedules them into a logically-equivalent
*canonical* circuit, annotates that, and transplants the detectors back onto the
original.

This notebook illustrates the three moving parts:

1. detecting that a schedule needs rescheduling,
2. rescheduling a round into canonical form, and
3. recovering the detectors (including for circuits with `REPEAT` blocks).

## 1. Rescheduling a single interleaved round

In the round below the ancilla `3` is reset late, sharing a moment with the
two-qubit gate `CX 0 3`. An `H` moment sits between the initial resets and that
late reset, so fragment splitting stops collecting leading resets before it
reaches `R 3` -- which is why the schedule needs rescheduling. Because `3` is
idle until its reset, the reset commutes to the round boundary, and
`canonicalize_collapsing_schedule` hoists it there to produce the canonical round
that fragment splitting expects.

In [ ]:
import stim

from tqecd.construction import (
    _annotate_clean_circuit,
    annotate_detectors_automatically,
)
from tqecd.schedule import (
    canonicalize_collapsing_schedule,
    fragment_schedule_needs_normalization,
)

interleaved_round = stim.Circuit("""
    QUBIT_COORDS(0, 0) 0
    QUBIT_COORDS(1, 0) 1
    QUBIT_COORDS(0, 1) 3
    R 0 1
    TICK
    H 0 1
    TICK
    R 3
    CX 0 3
    TICK
    CX 1 3
    TICK
    M 3
""")

print("needs rescheduling:", fragment_schedule_needs_normalization(interleaved_round))
print()
print(canonicalize_collapsing_schedule(interleaved_round))

## 2. A full memory experiment

Here is a distance-3 rotated-surface-code $Z$-memory that uses the interleaved
round above (the ancilla `13` is reset late). The naive fragment path drops the
interleaved reset and finds far fewer detectors than
`annotate_detectors_automatically`, whose output builds a valid (decomposable)
detector error model.

In [ ]:
se = """H 2 11 16 25\nTICK\nCX 2 3 16 17 11 12 15 14 10 9 19 18\nR 13\nTICK\nCX 2 1 16 15 11 10 8 14 3 9 12 18\nTICK\nCX 16 10 11 5 25 19 8 9 17 18 12 13\nTICK\nCX 16 8 11 3 25 17 1 9 10 18 5 13\nTICK\nH 2 11 16 25\nTICK"""
coords = """QUBIT_COORDS(1, 1) 1\nQUBIT_COORDS(2, 0) 2\nQUBIT_COORDS(3, 1) 3\nQUBIT_COORDS(5, 1) 5\nQUBIT_COORDS(1, 3) 8\nQUBIT_COORDS(2, 2) 9\nQUBIT_COORDS(3, 3) 10\nQUBIT_COORDS(4, 2) 11\nQUBIT_COORDS(5, 3) 12\nQUBIT_COORDS(6, 2) 13\nQUBIT_COORDS(0, 4) 14\nQUBIT_COORDS(1, 5) 15\nQUBIT_COORDS(2, 4) 16\nQUBIT_COORDS(3, 5) 17\nQUBIT_COORDS(4, 4) 18\nQUBIT_COORDS(5, 5) 19\nQUBIT_COORDS(4, 6) 25"""

init = stim.Circuit(
    coords + "\n"
    "R 1 3 5 8 10 12 15 17 19 2 9 11 14 16 18 25\n"
    "TICK\n" + se + "\nM 2 9 11 13 14 16 18 25\nTICK"
)
final = stim.Circuit(
    "R 2 9 11 14 16 18 25\nTICK\n"
    + se
    + "\nM 2 9 11 13 14 16 18 25 1 3 5 8 10 12 15 17 19"
)
circuit = init + final

naive = _annotate_clean_circuit(circuit)
annotated = annotate_detectors_automatically(circuit)
print("naive fragment path   :", naive.num_detectors, "detectors")
print("schedule-aware path   :", annotated.num_detectors, "detectors")

# The recovered detectors are deterministic and graphlike.
annotated.detector_error_model(decompose_errors=True)
print("detector error model builds -> detectors are consistent")

## 3. `REPEAT` blocks

The bulk rounds of a memory experiment are usually wrapped in a `REPEAT` block,
whose body may itself use an interleaved schedule. Rescheduling descends into the
loop body, keeps the loop structure, and transplants the detectors (and the
`SHIFT_COORDS` bookkeeping the annotator emits) back into the matching body.

In [ ]:
bulk = stim.Circuit(
    "R 2 9 11 14 16 18 25\nTICK\n" + se + "\nM 2 9 11 13 14 16 18 25\nTICK"
)
looped = init + bulk * 4 + final

annotated_looped = annotate_detectors_automatically(looped)
has_repeat = any(isinstance(inst, stim.CircuitRepeatBlock) for inst in annotated_looped)
print("REPEAT block preserved:", has_repeat)
print("detectors             :", annotated_looped.num_detectors)
annotated_looped.detector_error_model(decompose_errors=True)
print("detector error model builds -> detectors are consistent")